# Test de robustesse / overfitting du clustering K-Means

**Objectif** : vérifier que le clustering K-Means retenu (k=3, cf. `common.FORCED_K`) reflète
une vraie structure dans les données, et non un artefact de sur-ajustement à cet échantillon
précis. Cinq diagnostics indépendants sont menés :

1. **Stabilité par bootstrap** — le clustering est-il identique si on ré-échantillonne les retailers ?
2. **Stabilité du cluster à risque** — le segment de retailers à fort risque commercial (celui qui
   justifie de fixer k=3, cf. `decisions.FORCED_K_DECISION`) est-il stable sous ré-échantillonnage ?
3. **Sensibilité au bruit** — le clustering se dégrade-t-il progressivement (bon signe) ou
   brutalement (signe de sur-ajustement) quand on perturbe les features ?
4. **Sensibilité aux hyperparamètres** — le score de silhouette est-il stable autour de k=3
   pour différentes graines aléatoires ?
5. **Cohérence train/test** — un modèle entraîné sur `data_entrainement.csv` généralise-t-il à
   `data_test.csv` (jeu externe, jamais vu à l'entraînement) ?

> **Prérequis** : ce notebook doit être ouvert et exécuté avec `src/` comme dossier de travail
> (le même emplacement que l'ancien `Test_overfittinng.py`), pour que les chemins relatifs vers
> `../data/` et `../outputs/figures/` restent valides.


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

import Kmeans_recommendation as kmeans_mod
from common import RANDOM_STATE, identify_risk_cluster

import warnings
warnings.filterwarnings("ignore", module=r"^sklearn")
warnings.filterwarnings("ignore", module=r"^scipy")

%matplotlib inline

SCRIPT_DIR = Path.cwd()
TEST_DATA_PATH = SCRIPT_DIR / ".." / "data" / "data_test.csv"
FIGURES_DIR = SCRIPT_DIR / ".." / "outputs" / "figures"

N_BOOTSTRAP = 30
NOISE_LEVELS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
K_CANDIDATES = range(2, 9)


## 1. Chargement des données et détermination de k

On recharge le pipeline de préparation des features (`Kmeans_recommendation.load_and_prepare`)
et on retrouve le k retenu en production (`FORCED_K`, sinon l'optimum statistique par silhouette).


In [ ]:
df_retailers, pivot, X, preprocessor = kmeans_mod.load_and_prepare()

k_values, _, silhouettes = kmeans_mod.compute_elbow_and_silhouette(X)
auto_k_ref = k_values[int(np.argmax(silhouettes))]
k_ref = kmeans_mod.FORCED_K if kmeans_mod.FORCED_K is not None else auto_k_ref

print(f"k retenu pour les tests de robustesse : {k_ref} "
      f"({'forcé' if kmeans_mod.FORCED_K is not None else 'optimum statistique'}, "
      f"optimum statistique de référence : {auto_k_ref})")


## 2. Stabilité par bootstrap (Adjusted Rand Index)

On ré-échantillonne 80% des retailers, `N_BOOTSTRAP` fois, on reclusterise à chaque tirage, et on
compare (via l'ARI) le clustering obtenu à celui du modèle de référence entraîné sur l'ensemble des
données. Un ARI moyen proche de 1 indique un clustering stable ; proche de 0, un clustering instable
(probable sur-ajustement).


In [ ]:
def bootstrap_stability(X, k, n_bootstrap=N_BOOTSTRAP, sample_frac=0.8):
    reference_model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    reference_labels = reference_model.labels_
    n = X.shape[0]

    ari_scores = []
    rng = np.random.RandomState(RANDOM_STATE)
    for i in range(n_bootstrap):
        idx = rng.choice(n, size=int(n * sample_frac), replace=False)
        X_sample = X[idx]
        model = KMeans(n_clusters=k, n_init=10, random_state=i).fit(X_sample)
        sample_labels_pred = model.labels_
        sample_labels_ref = reference_labels[idx]
        ari_scores.append(adjusted_rand_score(sample_labels_ref, sample_labels_pred))

    return np.array(ari_scores)


def plot_bootstrap_distribution(ari_scores, out_path):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(ari_scores, bins=12, color="steelblue", edgecolor="white")
    ax.axvline(ari_scores.mean(), color="red", linestyle="--",
               label=f"ARI moyen = {ari_scores.mean():.3f}")
    ax.set_title(f"Stabilité des clusters par bootstrap ({N_BOOTSTRAP} ré-échantillonnages)")
    ax.set_xlabel("Adjusted Rand Index (vs. modèle de référence)")
    ax.set_ylabel("Fréquence")
    ax.legend()
    plt.tight_layout()
    os.makedirs(out_path.parent, exist_ok=True)
    plt.savefig(out_path, dpi=150)
    plt.show()


ari_scores = bootstrap_stability(X, k_ref)
plot_bootstrap_distribution(ari_scores, FIGURES_DIR / "bootstrap_stability.png")
print(f"ARI moyen : {ari_scores.mean():.3f} (écart-type : {ari_scores.std():.3f})")


## 3. Stabilité du cluster à risque (indice de Jaccard)

Le segment de retailers à très fort risque commercial (taux de forclusion/annulation élevés) est
la justification métier de forcer k=3 (cf. `decisions.FORCED_K_DECISION`). On vérifie ici que ce
cluster précis reste identifiable et stable sous ré-échantillonnage, via l'indice de Jaccard entre
le cluster de référence et son équivalent retrouvé dans chaque échantillon bootstrap.


In [ ]:
def risk_cluster_stability(X, k, reference_labels, risk_cluster_id,
                            n_bootstrap=N_BOOTSTRAP, sample_frac=0.8):
    reference_risk_members = set(np.where(reference_labels == risk_cluster_id)[0])
    n = X.shape[0]

    rng = np.random.RandomState(RANDOM_STATE)
    jaccard_scores = []
    no_match_count = 0

    for i in range(n_bootstrap):
        idx = rng.choice(n, size=int(n * sample_frac), replace=False)
        X_sample = X[idx]
        model = KMeans(n_clusters=k, n_init=10, random_state=i).fit(X_sample)
        sample_labels = model.labels_

        ref_risk_positions_in_sample = [
            pos for pos, original_idx in enumerate(idx) if original_idx in reference_risk_members
        ]

        if not ref_risk_positions_in_sample:
            jaccard_scores.append(np.nan)
            no_match_count += 1
            continue

        candidate_labels = sample_labels[ref_risk_positions_in_sample]
        matched_cluster = np.bincount(candidate_labels).argmax()

        predicted_members = set(np.where(sample_labels == matched_cluster)[0])
        reference_members = set(ref_risk_positions_in_sample)

        intersection = predicted_members & reference_members
        union = predicted_members | reference_members
        jaccard_scores.append(len(intersection) / len(union) if union else np.nan)

    return np.array(jaccard_scores), no_match_count


def plot_risk_cluster_stability(jaccard_scores, out_path):
    valid = jaccard_scores[~np.isnan(jaccard_scores)]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(valid, bins=12, color="firebrick", edgecolor="white")
    if valid.size:
        ax.axvline(valid.mean(), color="black", linestyle="--",
                   label=f"Jaccard moyen = {valid.mean():.3f}")
    ax.set_title(f"Stabilité du cluster à risque (Jaccard, {N_BOOTSTRAP} ré-échantillonnages)")
    ax.set_xlabel("Similarité de Jaccard vs. cluster à risque de référence")
    ax.set_ylabel("Fréquence")
    ax.set_xlim(-0.05, 1.05)
    ax.legend()
    plt.tight_layout()
    os.makedirs(out_path.parent, exist_ok=True)
    plt.savefig(out_path, dpi=150)
    plt.show()


reference_model = KMeans(n_clusters=k_ref, n_init=10, random_state=RANDOM_STATE).fit(X)
risk_cluster_id = identify_risk_cluster(df_retailers, reference_model.labels_)
jaccard_scores, no_match_count = risk_cluster_stability(X, k_ref, reference_model.labels_, risk_cluster_id)
plot_risk_cluster_stability(jaccard_scores, FIGURES_DIR / "risk_cluster_stability.png")

valid_jaccard = jaccard_scores[~np.isnan(jaccard_scores)]
print(f"Jaccard moyen : {valid_jaccard.mean():.3f} | "
      f"échantillons sans membre du cluster à risque : {no_match_count}/{N_BOOTSTRAP}")


## 4. Sensibilité au bruit

On ajoute un bruit gaussien croissant aux features, on reclusterise, et on mesure l'ARI par rapport
au clustering sans bruit. Une dégradation progressive et régulière est attendue (bon signe) ; une
chute brutale au moindre bruit indiquerait un clustering instable, sensible au sur-ajustement.


In [ ]:
def noise_sensitivity(X, k, noise_levels=NOISE_LEVELS):
    reference_model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    reference_labels = reference_model.labels_

    rng = np.random.RandomState(RANDOM_STATE)
    results = []
    for noise_std in noise_levels:
        noisy_X = X + rng.normal(0, noise_std, size=X.shape)
        noisy_labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(noisy_X)
        ari = adjusted_rand_score(reference_labels, noisy_labels)
        results.append(ari)
    return results


def plot_noise_sensitivity(noise_levels, ari_scores, out_path):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(noise_levels, ari_scores, "o-", color="darkorange")
    ax.set_title("Sensibilité au bruit (dégradation de l'ARI)")
    ax.set_xlabel("Écart-type du bruit gaussien ajouté")
    ax.set_ylabel("ARI vs. clustering sans bruit")
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    os.makedirs(out_path.parent, exist_ok=True)
    plt.savefig(out_path, dpi=150)
    plt.show()


noise_aris = noise_sensitivity(X, k_ref)
plot_noise_sensitivity(NOISE_LEVELS, noise_aris, FIGURES_DIR / "noise_sensitivity.png")
for level, ari in zip(NOISE_LEVELS, noise_aris):
    print(f"bruit std={level:.2f} -> ARI={ari:.3f}")


## 5. Sensibilité aux hyperparamètres (choix de k)

Pour chaque k candidat (2 à 8), on recalcule le score de silhouette sur 5 graines aléatoires
différentes. Un écart-type faible autour de k=3 indique que le choix de k n'est pas un artefact
d'initialisation hasardeuse.


In [ ]:
def hyperparameter_sensitivity(X, k_candidates=K_CANDIDATES, n_seeds=5):
    results = {}
    for k in k_candidates:
        scores = []
        for seed in range(n_seeds):
            labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(X)
            scores.append(silhouette_score(X, labels))
        results[k] = {"mean": np.mean(scores), "std": np.std(scores)}
    return results


def plot_hyperparameter_sensitivity(results, out_path):
    ks = list(results.keys())
    means = [results[k]["mean"] for k in ks]
    stds = [results[k]["std"] for k in ks]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(ks, means, yerr=stds, fmt="o-", color="seagreen", capsize=4)
    ax.set_title("Stabilité du score de silhouette selon k (5 seeds par k)")
    ax.set_xlabel("Nombre de clusters (k)")
    ax.set_ylabel("Score de silhouette (moyenne ± écart-type)")
    plt.tight_layout()
    os.makedirs(out_path.parent, exist_ok=True)
    plt.savefig(out_path, dpi=150)
    plt.show()


hp_results = hyperparameter_sensitivity(X)
plot_hyperparameter_sensitivity(hp_results, FIGURES_DIR / "hyperparameter_sensitivity.png")
for k, stats in hp_results.items():
    print(f"k={k} -> silhouette moyenne={stats['mean']:.3f} (± {stats['std']:.3f})")


## 6. Cohérence train/test (généralisation à un jeu externe)

Dernier test, le plus direct : un modèle entraîné sur `data_entrainement.csv` produit-il des
affectations de cluster cohérentes avec un modèle entraîné directement sur `data_test.csv`
(jeu jamais vu à l'entraînement) ? Un ARI élevé confirme que le clustering généralise, et pas
seulement qu'il colle à l'échantillon d'entraînement.


In [ ]:
def load_external_test_set(preprocessor, test_csv_path=TEST_DATA_PATH):
    df_raw_test = kmeans_mod.load_raw_data(str(test_csv_path))
    df_test_retailers = kmeans_mod.build_retailer_features(df_raw_test)
    pivot_test = kmeans_mod.build_retailer_game_matrix(df_raw_test)

    X_test = preprocessor.transform(
        df_test_retailers[kmeans_mod.NUMERIC_FEATURES + kmeans_mod.CATEGORICAL_FEATURES]
    )
    if hasattr(X_test, "toarray"):
        X_test = X_test.toarray()

    return df_test_retailers, pivot_test, X_test


def train_test_consistency(X_train, X_test, k):
    model_train = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_train)
    labels_test_predicted = model_train.predict(X_test)

    model_test_direct = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_test)
    labels_test_direct = model_test_direct.labels_

    return adjusted_rand_score(labels_test_predicted, labels_test_direct)


def full_train_test_validation(df_retailers, pivot, X, preprocessor, k):
    df_test, pivot_test, X_test = load_external_test_set(preprocessor)

    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    labels_train = model.labels_
    labels_test = model.predict(X_test)

    ari = train_test_consistency(X, X_test, k)

    return {"labels_train": labels_train, "labels_test": labels_test, "ari": ari}


result = full_train_test_validation(df_retailers, pivot, X, preprocessor, k_ref)
print(f"ARI train->test (cohérence de généralisation) : {result['ari']:.3f}")


## Synthèse

Relire les 5 scores ci-dessus (ARI bootstrap, Jaccard cluster à risque, ARI bruit, écart-type
silhouette par k, ARI train/test) pour conclure si k=3 constitue une structure stable et
généralisable, ou si certains signaux (scores bas, forte variance) méritent d'être documentés
dans `src/decisions.py` avant validation métier.
